# Fungsi Aktivasi (Activation Functions) dalam Deep Learning

**Kursus:** NCA-GENL — Navasena Generative ML Course  
**Modul:** 02 — Deep Learning Fundamentals  
**Estimasi sesi:** ± 40 menit di T4 (termasuk membaca)  
**Modul 2: Deep Learning Fundamentals** | Notebook 2 dari 6

---

## Tujuan Pembelajaran (Learning Objectives)

Setelah menyelesaikan notebook ini, kamu akan mampu:

1. **Memahami peran fungsi aktivasi** — mengapa neural network TIDAK bisa belajar tanpa fungsi aktivasi
2. **Mengenal fungsi aktivasi populer** — Sigmoid, Tanh, ReLU, Leaky ReLU, ELU, Swish, dan Softmax
3. **Memilih aktivasi yang tepat** — panduan praktis untuk berbagai jenis masalah
4. **Membandingkan performa** — melihat langsung dampak pilihan aktivasi pada akurasi model
5. **Memahami peran optimizer** — bagaimana Adam, SGD, dan RMSprop berbeda

---

> **Prasyarat:** Sudah menyelesaikan Notebook 1 (Neural Network Dasar dengan Fashion MNIST)

In [ ]:
import os
import tensorflow as tf
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_style('whitegrid')
tf.random.set_seed(42)
np.random.seed(42)

QUICK = bool(os.environ.get('QUICK'))
EPOCHS = 2 if QUICK else 10

print(f"TensorFlow version: {tf.__version__}")
print(f"Mode QUICK: {QUICK} (EPOCHS per eksperimen = {EPOCHS})")
if tf.config.list_physical_devices('GPU'):
    print("✅ GPU terdeteksi!")
else:
    print("⚠️ Tidak ada GPU — tetap bisa jalan.")

---

## Bagian 1: Mengapa Perlu Fungsi Aktivasi?

### Kenapa aktivasi dibutuhkan

Bayangkan neural network tanpa fungsi aktivasi seperti **kalkulator biasa** — hanya bisa menjumlahkan dan mengalikan. Fungsi aktivasi adalah yang membuat neural network **bisa mempelajari pola melengkung**, bukan hanya garis lurus.

> **Fakta penting:** Tanpa fungsi aktivasi, 100 layer pun sama saja dengan 1 layer — hanya transformasi linear!

Yang membedakan satu aktivasi dari yang lain adalah **bentuk fungsinya dan turunannya**, bukan seberapa "pintar" ia terdengar:
- **ReLU** menolkan seluruh masukan negatif dan meneruskan yang positif apa adanya, jadi turunannya rata 1 di sisi positif
- **Sigmoid** memampatkan masukan apa pun ke rentang 0–1, dan turunannya mendekati nol di kedua ujung
- **Tanh** memampatkan ke rentang −1–1, jadi keluarannya berpusat di nol — itu bedanya dengan Sigmoid

Bentuk-bentuk inilah yang kita plot di bagian berikutnya, lalu kita bandingkan dampaknya pada akurasi.

In [ ]:
# Demo: Tanpa Aktivasi = Tetap Linear
np.random.seed(42)

# Simulasi 2 layer tanpa aktivasi
W1 = np.array([[2, 3], [1, 4]])  # Weight layer 1
W2 = np.array([[5, 1], [2, 3]])  # Weight layer 2

# Input
x_demo = np.array([1, 2])

# Tanpa aktivasi: layer1 -> layer2
layer1 = W1 @ x_demo
layer2 = W2 @ layer1
print(f"2 layer tanpa aktivasi: {layer2}")

# Sama saja dengan 1 layer!
W_gabungan = W2 @ W1
satu_layer = W_gabungan @ x_demo
print(f"1 layer gabungan     : {satu_layer}")
print("\nHasil sama persis — tanpa aktivasi, layer tambahan tidak menambah kemampuan model.")

Inilah mengapa fungsi aktivasi sangat penting — mereka menambahkan **non-linearitas** yang memungkinkan neural network mempelajari pola kompleks seperti pengenalan gambar, teks, dan suara.

---

## Bagian 2: Fungsi Aktivasi Utama

Berikut fungsi-fungsi aktivasi yang paling sering digunakan, dari yang paling sederhana hingga yang paling modern.

In [ ]:
# Buat data input
x = np.linspace(-5, 5, 200)

# Hitung semua fungsi aktivasi
activations = {
    'Sigmoid': tf.keras.activations.sigmoid(x).numpy(),
    'Tanh': tf.keras.activations.tanh(x).numpy(),
    'ReLU': tf.keras.activations.relu(x).numpy(),
    'Leaky ReLU': tf.nn.leaky_relu(x, alpha=0.1).numpy(),
    'ELU': tf.keras.activations.elu(x).numpy(),
    'Swish': tf.keras.activations.swish(x).numpy(),
}
print(f"✅ {len(activations)} fungsi aktivasi siap divisualisasikan!")

### Ringkasan Fungsi Aktivasi

| Fungsi | Formula | Output Range | Kelebihan | Kekurangan |
|--------|---------|-------------|-----------|------------|
| **Sigmoid** | 1/(1+e^(-x)) | 0 – 1 | Cocok untuk probabilitas | Vanishing gradient |
| **Tanh** | (e^x - e^(-x))/(e^x + e^(-x)) | -1 – 1 | Zero-centered | Vanishing gradient |
| **ReLU** | max(0, x) | 0 – ∞ | Cepat, sederhana | Dead neurons |
| **Leaky ReLU** | max(0.1x, x) | -∞ – ∞ | Mengatasi dead neurons | Sedikit lebih lambat |
| **ELU** | x jika x>0, α(e^x-1) jika x≤0 | ~-1 – ∞ | Smooth, zero-mean | Lebih lambat |
| **Swish** | x × sigmoid(x) | ~-0.3 – ∞ | Smooth, performa bagus | Lebih lambat |

**Istilah penting:**
- **Vanishing gradient** = gradient menjadi sangat kecil saat backpropagation (propagasi mundur), membuat model sulit belajar
- **Dead neurons** = neuron yang selalu menghasilkan 0 dan berhenti belajar
- **Zero-centered** = output rata-rata mendekati 0, membantu proses training lebih stabil

In [ ]:
colors = ['#E53935', '#FB8C00', '#43A047', '#1E88E5', '#8E24AA', '#00ACC1']

fig, axes = plt.subplots(2, 3, figsize=(15, 9))

for ax, (name, y), color in zip(axes.flat, activations.items(), colors):
    ax.plot(x, y, color=color, linewidth=2.5, label=name)
    ax.axhline(y=0, color='gray', linewidth=0.5, linestyle='--')
    ax.axvline(x=0, color='gray', linewidth=0.5, linestyle='--')
    ax.set_title(name, fontsize=14, fontweight='bold')
    ax.set_xlabel('Input (x)')
    ax.set_ylabel('Output')
    ax.grid(True, alpha=0.3)
    ax.set_xlim(-5, 5)

plt.suptitle('Fungsi Aktivasi dalam Deep Learning', fontsize=16, fontweight='bold')
plt.tight_layout()
plt.show()

### Perbandingan Langsung

Semua fungsi aktivasi ditumpuk (overlay) dalam satu grafik supaya perbedaannya lebih jelas.

In [ ]:
plt.figure(figsize=(12, 6))
for (name, y), color in zip(activations.items(), colors):
    plt.plot(x, y, label=name, linewidth=2.5, color=color)

plt.axhline(y=0, color='gray', linewidth=0.5, linestyle='--')
plt.axvline(x=0, color='gray', linewidth=0.5, linestyle='--')
plt.xlabel('Input (x)', fontsize=12)
plt.ylabel('Output', fontsize=12)
plt.title('Perbandingan Semua Fungsi Aktivasi', fontsize=14, fontweight='bold')
plt.legend(fontsize=11, loc='upper left')
plt.grid(True, alpha=0.3)
plt.xlim(-5, 5)
plt.ylim(-2, 5)
plt.tight_layout()
plt.show()

---

## Bagian 3: Softmax — Aktivasi Khusus untuk Klasifikasi

### Softmax — Mengubah Angka Menjadi Probabilitas

Softmax berbeda dari aktivasi lain — ia bekerja pada **SELURUH output layer sekaligus**.

Bayangkan kamu di restoran dan harus memilih 1 dari 5 menu. Softmax mengubah 'skor' setiap menu menjadi **probabilitas**, dan totalnya selalu **100%**.

**Formula Softmax:**

$$\text{softmax}(x_i) = \frac{e^{x_i}}{\sum_{j} e^{x_j}}$$

Setiap nilai diperkuat secara eksponensial, lalu dibagi total — sehingga nilai tertinggi mendapat probabilitas terbesar, dan semua probabilitas berjumlah 1.

In [ ]:
# Contoh: output mentah dari neural network (logits)
logits = tf.constant([[4.0, 2.0, 0.5, -1.0, 0.1]])
probabilities = tf.keras.activations.softmax(logits)

categories = ['T-shirt', 'Celana', 'Pullover', 'Dress', 'Coat']

print("Skor mentah (logits) vs Probabilitas (softmax):\n")
for cat, logit, prob in zip(categories, logits[0].numpy(), probabilities[0].numpy()):
    bar = '█' * int(prob * 50)
    print(f"  {cat:10s}  logit={logit:5.1f}  →  prob={prob:.1%}  {bar}")

print(f"\n  Total probabilitas: {probabilities[0].numpy().sum():.1%} ✅")

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))

ax1.barh(categories, logits[0].numpy(), color='#90CAF9', edgecolor='#1565C0')
ax1.set_xlabel('Skor')
ax1.set_title('Sebelum Softmax (Logits)', fontweight='bold')
ax1.axvline(x=0, color='gray', linewidth=0.5)

colors_bar = ['#43A047' if i == 0 else '#90CAF9' for i in range(5)]
ax2.barh(categories, probabilities[0].numpy(), color=colors_bar, edgecolor='#1565C0')
ax2.set_xlabel('Probabilitas')
ax2.set_title('Sesudah Softmax (Probabilitas)', fontweight='bold')
ax2.set_xlim(0, 1)

plt.suptitle('Softmax: Mengubah Skor → Probabilitas', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

---

## Bagian 4: Panduan Memilih Aktivasi

## Kapan Menggunakan Aktivasi Apa?

Berikut panduan praktis yang digunakan para profesional:

| Situasi | Aktivasi yang Direkomendasikan | Alasan |
|---------|------------------------------|--------|
| Hidden layers (default) | **ReLU** | Cepat, efektif, standar industri |
| Hidden layers (advanced) | **Swish** atau **Leaky ReLU** | Performa sedikit lebih baik |
| Output — Klasifikasi biner | **Sigmoid** | Output 0-1 = probabilitas Ya/Tidak |
| Output — Klasifikasi multi-kelas | **Softmax** | Probabilitas untuk setiap kelas |
| Output — Regresi | **Linear (None)** | Prediksi angka bebas |

> **Tips dari industri:** Mulai dengan ReLU di hidden layers. Jika ada masalah (dead neurons, akurasi tidak naik), coba Leaky ReLU atau Swish.

### Diagram Keputusan Memilih Fungsi Aktivasi

```
Memilih Fungsi Aktivasi:
┌─ Output layer?
│   ├─ Ya/Tidak (biner)  → Sigmoid
│   ├─ Banyak kelas      → Softmax
│   └─ Angka (regresi)   → Linear/None
└─ Hidden layer?
    ├─ Default            → ReLU
    ├─ Ada dead neurons   → Leaky ReLU
    └─ Perlu performa max → Swish
```

**Kapan ada dead neurons (neuron mati)?**  
Ketika banyak neuron selalu menghasilkan output 0 dan tidak pernah aktif. Ini terjadi saat learning rate terlalu besar atau data tidak ternormalisasi. Solusi: gunakan **Leaky ReLU** yang tetap mengalirkan gradient kecil untuk input negatif.

---

## Bagian 5: Eksperimen Praktis — Dampak Aktivasi pada Training

## Eksperimen: Bagaimana Aktivasi Mempengaruhi Hasil?

Model Fashion MNIST yang sama dilatih dengan tiap fungsi aktivasi, lalu hasilnya dibandingkan langsung.

Kita akan menggunakan dataset Fashion MNIST (sama seperti Notebook 1) — 70.000 gambar pakaian dalam 10 kategori.

**Metode perbandingan:** setiap varian dinilai dari `val_accuracy` di epoch terakhir (bukan data test) — data test hanya disentuh sekali di akhir, untuk aktivasi yang terpilih sebagai pemenang. Estimasi waktu: ± 3–5 menit di T4 untuk 6 model kecil (5 aktivasi + 1 baseline linear).

In [ ]:
# Load Fashion MNIST
(X_train_full, y_train_full), (X_test, y_test) = tf.keras.datasets.fashion_mnist.load_data()

if QUICK:
    X_train_full, y_train_full = X_train_full[:8000], y_train_full[:8000]
    X_test, y_test = X_test[:2000], y_test[:2000]

N_VALID = 1000 if QUICK else 5000
X_train, y_train = X_train_full[:-N_VALID] / 255.0, y_train_full[:-N_VALID]
X_valid, y_valid = X_train_full[-N_VALID:] / 255.0, y_train_full[-N_VALID:]
X_test = X_test / 255.0

print(f"Data training: {X_train.shape[0]:,} gambar")
print(f"Data validation: {X_valid.shape[0]:,} gambar")
print(f"Data testing : {X_test.shape[0]:,} gambar")

### Baseline: Model Linear (Tanpa Aktivasi)

Sebelum membandingkan macam-macam aktivasi, kita cek dulu performa model yang hidden layer-nya TIDAK memakai aktivasi sama sekali (`activation=None`, transformasi linear murni). Baseline ini adalah versi nyata dari demo numpy di Bagian 1: kalau argumennya benar, model ini seharusnya kalah jauh dari model dengan aktivasi asli.

In [ ]:
def latih_dengan_aktivasi(nama_aktivasi, activation_fn, epochs=EPOCHS):
    """Latih model dengan fungsi aktivasi tertentu.
    Kembalikan model, history, dan val_accuracy di epoch terakhir."""
    model = tf.keras.Sequential([
        tf.keras.Input(shape=(28, 28)),
        tf.keras.layers.Flatten(),
        tf.keras.layers.Dense(128, activation=activation_fn),
        tf.keras.layers.Dense(64, activation=activation_fn),
        tf.keras.layers.Dense(10, activation='softmax')
    ])
    model.compile(loss='sparse_categorical_crossentropy', optimizer='adam', metrics=['accuracy'])

    history = model.fit(X_train, y_train, epochs=epochs,
                        validation_data=(X_valid, y_valid), verbose=0)

    val_acc = history.history['val_accuracy'][-1]
    print(f"  {nama_aktivasi:20s} -> val_accuracy: {val_acc:.1%}")
    return model, history, val_acc

baseline_model, baseline_history, baseline_val_acc = latih_dengan_aktivasi('Linear (baseline)', None)

In [ ]:
print("Melatih model dengan berbagai fungsi aktivasi...\n")

results = {}
for name, fn in [('ReLU', 'relu'), ('Sigmoid', 'sigmoid'), ('Tanh', 'tanh'),
                 ('Leaky ReLU', tf.keras.layers.LeakyReLU()), ('Swish', 'swish')]:
    model, h, val_acc = latih_dengan_aktivasi(name, fn)
    results[name] = {'model': model, 'history': h, 'val_accuracy': val_acc}

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

# Bar chart akurasi
names = list(results.keys())
accs = [results[n]['val_accuracy'] for n in names]
bar_colors = ['#43A047', '#E53935', '#FB8C00', '#1E88E5', '#8E24AA']
bars = ax1.bar(names, accs, color=bar_colors, edgecolor='white', linewidth=2)
for bar, acc in zip(bars, accs):
    ax1.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 0.002,
             f'{acc:.1%}', ha='center', va='bottom', fontweight='bold')
ax1.set_ylim(min(accs) - 0.03, max(accs) + 0.03)
ax1.set_ylabel('Val Accuracy (epoch terakhir)')
ax1.set_title('Perbandingan Val Accuracy per Aktivasi', fontweight='bold')
ax1.grid(axis='y', alpha=0.3)

# Training curves
for name, color in zip(names, bar_colors):
    ax2.plot(results[name]['history'].history['val_accuracy'],
             label=name, linewidth=2, color=color)
ax2.set_xlabel('Epoch')
ax2.set_ylabel('Validation Accuracy')
ax2.set_title('Kurva Belajar per Aktivasi', fontweight='bold')
ax2.legend()
ax2.grid(True, alpha=0.3)

plt.suptitle('Eksperimen: Dampak Fungsi Aktivasi', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

### Pemenang dan Evaluasi Test (Sekali)

In [ ]:
aktivasi_terbaik = max(results, key=lambda n: results[n]['val_accuracy'])
model_terbaik = results[aktivasi_terbaik]['model']
test_loss, test_acc = model_terbaik.evaluate(X_test, y_test, verbose=0)

print(f"Baseline linear (tanpa aktivasi hidden layer): val_accuracy {baseline_val_acc:.1%}")
print(f"Aktivasi terbaik (val_accuracy epoch terakhir): {aktivasi_terbaik}")
print(f"Selisih terhadap baseline: {(results[aktivasi_terbaik]['val_accuracy'] - baseline_val_acc) * 100:.1f} pp")
print(f"Akurasi test model {aktivasi_terbaik} (test set disentuh sekali): {test_acc:.1%}")

---

## Bagian 6: Optimizer — Strategi Belajar Neural Network

## Bonus: Optimizer — Cara Neural Network Belajar

Selain fungsi aktivasi, **optimizer** juga sangat penting. Optimizer menentukan **BAGAIMANA** model memperbaiki kesalahannya.

### Analogi: Turun Gunung dalam Kabut

Bayangkan kamu tersesat di gunung berkabut dan ingin turun ke lembah (titik terendah = loss minimum):

- **SGD** = melangkah menurut kemiringan saat itu saja; jalurnya bisa bolak-balik menyeberangi lembah dan butuh lebih banyak langkah.
- **SGD + Momentum** = Seperti bola menggelinding — punya inersia, lebih cepat.
- **RMSprop** = Menyesuaikan ukuran langkah berdasarkan medan — berguna di lereng curam.
- **Adam** = Seperti GPS yang menyesuaikan arah DAN kecepatan langkah secara adaptif. Titik awal yang wajar, bukan jaminan hasil terbaik.

**Adam** adalah singkatan dari **Adaptive Moment Estimation** — menggabungkan keunggulan Momentum dan RMSprop.

**Metode perbandingan:** sama seperti eksperimen aktivasi — 4 optimizer dinilai dari `val_accuracy` epoch terakhir, test set disentuh sekali di akhir untuk optimizer terpilih. Estimasi waktu: ± 2–4 menit di T4 untuk 4 model kecil.

### Perbandingan Optimizer

| Optimizer | Kecepatan | Kestabilan | Kapan Digunakan |
|-----------|----------|-----------|----------------|
| **SGD** | Lambat | Stabil | Riset, fine-tuning |
| **SGD + Momentum** | Sedang | Stabil | Training CNN besar |
| **Adam** | Cepat | Sangat baik | Default untuk kebanyakan kasus |
| **RMSprop** | Cepat | Baik | RNN/LSTM |

> **Rekomendasi praktis:** Mulai selalu dengan **Adam**. Jika training tidak stabil, coba SGD dengan momentum.

In [ ]:
def latih_dengan_optimizer(nama, optimizer, epochs=EPOCHS):
    """Latih model dengan optimizer tertentu.
    Kembalikan model, history, dan val_accuracy di epoch terakhir."""
    model = tf.keras.Sequential([
        tf.keras.Input(shape=(28, 28)),
        tf.keras.layers.Flatten(),
        tf.keras.layers.Dense(128, activation='relu'),
        tf.keras.layers.Dense(64, activation='relu'),
        tf.keras.layers.Dense(10, activation='softmax')
    ])
    model.compile(loss='sparse_categorical_crossentropy', optimizer=optimizer, metrics=['accuracy'])

    history = model.fit(X_train, y_train, epochs=epochs,
                        validation_data=(X_valid, y_valid), verbose=0)

    val_acc = history.history['val_accuracy'][-1]
    print(f"  {nama:15s} -> val_accuracy: {val_acc:.1%}")
    return model, history, val_acc

print("Melatih model dengan berbagai optimizer...\n")

opt_results = {}
optimizers_list = [
    ('SGD', 'sgd'),
    ('SGD + Momentum', tf.keras.optimizers.SGD(momentum=0.9)),
    ('Adam', 'adam'),
    ('RMSprop', 'rmsprop'),
]

for name, opt in optimizers_list:
    model, h, val_acc = latih_dengan_optimizer(name, opt)
    opt_results[name] = {'model': model, 'history': h, 'val_accuracy': val_acc}

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

opt_names = list(opt_results.keys())
opt_accs = [opt_results[n]['val_accuracy'] for n in opt_names]
opt_colors = ['#EF5350', '#FF7043', '#66BB6A', '#42A5F5']

bars = ax1.bar(opt_names, opt_accs, color=opt_colors, edgecolor='white', linewidth=2)
for bar, acc in zip(bars, opt_accs):
    ax1.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 0.002,
             f'{acc:.1%}', ha='center', va='bottom', fontweight='bold')
ax1.set_ylim(min(opt_accs) - 0.03, max(opt_accs) + 0.03)
ax1.set_ylabel('Val Accuracy (epoch terakhir)')
ax1.set_title('Perbandingan Val Accuracy per Optimizer', fontweight='bold')
ax1.grid(axis='y', alpha=0.3)

for name, color in zip(opt_names, opt_colors):
    ax2.plot(opt_results[name]['history'].history['val_accuracy'],
             label=name, linewidth=2, color=color)
ax2.set_xlabel('Epoch')
ax2.set_ylabel('Validation Accuracy')
ax2.set_title('Kecepatan Belajar per Optimizer', fontweight='bold')
ax2.legend()
ax2.grid(True, alpha=0.3)

plt.suptitle('Eksperimen: Dampak Optimizer', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

### Pemenang dan Evaluasi Test (Sekali)

In [ ]:
optimizer_terbaik = max(opt_results, key=lambda n: opt_results[n]['val_accuracy'])
model_terbaik_opt = opt_results[optimizer_terbaik]['model']
test_loss_opt, test_acc_opt = model_terbaik_opt.evaluate(X_test, y_test, verbose=0)

print(f"Optimizer terbaik (val_accuracy epoch terakhir): {optimizer_terbaik}")
print(f"Akurasi test model {optimizer_terbaik} (test set disentuh sekali): {test_acc_opt:.1%}")

---

## Bagian 7: Peran NVIDIA dalam Optimasi Deep Learning

## Peran NVIDIA dalam Optimasi Deep Learning

NVIDIA tidak hanya membuat GPU untuk gaming — mereka adalah tulang punggung deep learning modern:

- **cuDNN** (CUDA Deep Neural Network library) mengoptimasi komputasi aktivasi di GPU — ReLU, Sigmoid, Tanh semuanya diakselerasi secara otomatis
- **TensorRT** (NVIDIA inference optimizer) secara otomatis memilih dan menggabungkan (fuse) aktivasi untuk kecepatan maksimal saat deployment
- Framework TensorFlow, PyTorch, dll sudah terintegrasi dengan cuDNN sehingga aktivasi otomatis berjalan di GPU tanpa konfigurasi tambahan
- **Mixed Precision Training** (pelatihan presisi campuran): NVIDIA Tensor Cores memungkinkan training dengan campuran float16/float32, mempercepat 2-3x dengan memori lebih hemat

**Mengapa ini penting?** Sebuah model dengan 128 juta parameter (seperti BERT) perlu menghitung aktivasi untuk setiap neuron di setiap layer untuk setiap data. Di CPU, ini bisa memakan waktu berjam-jam. Di GPU NVIDIA dengan cuDNN, bisa selesai dalam menit!

In [ ]:
if tf.config.list_physical_devices('GPU'):
    print("🔲 Informasi GPU NVIDIA:")
    gpu = tf.config.list_physical_devices('GPU')[0]
    print(f"   Device: {gpu}")
    print(f"   cuDNN enabled: {tf.test.is_built_with_cuda()}")
    print(f"\n   cuDNN mengoptimasi semua fungsi aktivasi yang kita pelajari hari ini!")
    print(f"   Di GPU, komputasi aktivasi jauh lebih cepat; seberapa cepat, kita ukur sendiri di Notebook 5.")
else:
    print("💡 Di Google Colab, aktifkan GPU untuk merasakan percepatan NVIDIA:")
    print("   Runtime → Change runtime type → T4 GPU")
    print("\n   NVIDIA cuDNN mengoptimasi fungsi aktivasi di GPU — ")
    print("   ReLU, Sigmoid, Tanh semuanya diakselerasi secara otomatis!")

## 🏋️ Latihan

1. Tambahkan ELU (`tf.keras.activations.elu`) ke daftar aktivasi di Bagian 5, latih ulang, lalu lihat di posisi mana `val_accuracy` ELU dibanding 5 aktivasi lain.
2. Ganti learning rate Adam dari default menjadi 0.01, lalu 0.0001 (`tf.keras.optimizers.Adam(learning_rate=...)`), latih ulang model ReLU dari Bagian 5, dan bandingkan `val_accuracy`-nya dengan Adam default.
3. Gabungkan aktivasi pemenang dan optimizer pemenang dari dua eksperimen di atas dalam satu model baru, lalu bandingkan `val_accuracy`-nya dengan model baseline linear di awal Bagian 5.

**Petunjuk:** fungsi `latih_dengan_aktivasi` dan `latih_dengan_optimizer` bisa dipanggil ulang dengan parameter baru — tidak perlu menulis ulang dari nol.

In [ ]:
# TODO: tulis kodemu di sini

---

## Kesimpulan

Selamat! Kamu telah menyelesaikan notebook tentang Fungsi Aktivasi. Berikut yang sudah kita pelajari:

- ✅ **Memahami mengapa fungsi aktivasi diperlukan** — tanpa aktivasi, neural network hanya bisa melakukan transformasi linear, tidak peduli berapa banyak layer
- ✅ **Mengenal 6+ fungsi aktivasi** — Sigmoid, Tanh, ReLU, Leaky ReLU, ELU, Swish, dan Softmax beserta kelebihan/kekurangan masing-masing
- ✅ **Softmax untuk klasifikasi multi-kelas** — mengubah skor mentah (logits) menjadi probabilitas yang berjumlah 100%
- ✅ **Membandingkan dampak aktivasi dan optimizer** — secara eksperimental membuktikan bahwa pilihan aktivasi dan optimizer mempengaruhi performa
- ✅ **Panduan praktis memilih aktivasi dan optimizer** — ReLU untuk hidden layers, Adam untuk optimizer default

### Ringkasan Rekomendasi

| Komponen | Pilihan Default | Alternatif |
|----------|----------------|------------|
| Hidden layer activation | **ReLU** | Swish, Leaky ReLU |
| Output biner | **Sigmoid** | — |
| Output multi-kelas | **Softmax** | — |
| Optimizer | **Adam** | SGD+Momentum |

---

🔜 **Selanjutnya: Notebook 3 — CNN (Convolutional Neural Network) untuk Klasifikasi Gambar**

Di sana kita akan belajar bagaimana neural network bisa "melihat" dan mengenali gambar menggunakan operasi konvolusi (convolution) — teknik yang digunakan di balik Face ID, kamera HP modern, dan mobil self-driving!